# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset on second primary colorectal cancer in survivors, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities will be referenced by their `@id` to ensure reproducibility and clarity.

### Dataset Source

The Croissant schema is accessed via:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the dataset metadata and inspect its basic information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Published:", metadata.datePublished)
print("Cite as:", metadata.citeAs)

## 2. Data Overview

List the available record sets, their `@id`s, and summarize associated fields and columns. All references are made via `@id` in the Croissant metadata.

*Note: Record sets contain the primary data tables; fields/columns define the variables for each record set.*

In [ ]:
from collections import defaultdict
# List all record sets and their fields using their `@id`

record_sets = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # metadata.recordSet is a list of RecordSet objects
    print(f"Found {len(metadata.recordSet)} record set(s):\n")
    for rs in metadata.recordSet:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs['@id'] if '@id' in rs else getattr(rs, '@id', None)}")
        record_sets.append(rs['@id'] if '@id' in rs else getattr(rs, '@id', None))
        # List fields
        if hasattr(rs, 'field') and rs.field:
            print("  Fields:")
            for f in rs.field:
                f_id = f['@id'] if '@id' in f else getattr(f, '@id', None)
                print(f"    - name: {getattr(f, 'name', None)} (@id: {f_id})")
        # List columns
        if hasattr(rs, 'column') and rs.column:
            print("  Columns:")
            for c in rs.column:
                c_id = c['@id'] if '@id' in c else getattr(c, '@id', None)
                print(f"    - name: {getattr(c, 'name', None)} (@id: {c_id})")
        print()
else:
    print("No record sets were found in the Croissant metadata.")

## 3. Data Extraction

Extract data from each record set using its `@id`, loading into a Pandas DataFrame for further analysis.

In [ ]:
# If no record sets are present, we cannot proceed. Otherwise, extract!
if not record_sets:
    print("No record sets available to extract data.")
else:
    dataframes = {}
    for rs_id in record_sets:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set {rs_id} with {len(df)} records and columns:")
            print(df.columns.tolist())
            display(df.head(3))
        except Exception as e:
            print(f"Error extracting {rs_id}: {e}")

# For demonstration, select the first record set for further exploration
if record_sets:
    main_rs_id = record_sets[0]
    main_df = dataframes[main_rs_id]
    print(f"\n---\nMain DataFrame loaded from RecordSet @id: {main_rs_id}")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

Explore, filter and process data from the main record set. We'll select a numeric field and a group field using their `@id`.
Remember to refer to variable/field names by their column keys, which originate from the Croissant `@id` field mappings.

In [ ]:
# Identify numeric and categorical fields from the DataFrame
if record_sets:
    # For demonstration, infer numeric fields (e.g., age, diagnosis interval, etc.)
    display(main_df.head())
    print("\nDataFrame columns:")
    print(main_df.columns.tolist())
    
    # Attempt to select a numeric field; adjust as per dataset's columns
    possible_numeric_fields = [col for col in main_df.columns if main_df[col].dtype.kind in {'i', 'u', 'f'} or 'age' in col.lower() or 'interval' in col.lower()]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
    else:
        print("No obvious numeric fields found; EDA may be limited.")
        numeric_field = main_df.columns[0]  # fallback
    print(f"Chosen numeric field (@id or column): {numeric_field}")

    # Select a threshold for this field (example: mean value)
    try:
        threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else None
        if threshold is None:
            # Fallback for non-numeric field
            threshold = 0
        filtered_df = main_df[main_df[numeric_field] > threshold] if threshold is not None else main_df.copy()
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization step
        if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
            filtered_df = filtered_df.copy()
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            )
            print(f"\nNormalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print(f"{numeric_field} is not numeric and cannot be normalized.")

        # Grouping
        # Choose a categorical field (e.g. with 'group', 'site', or similar in name)
        possible_group_fields = [col for col in main_df.columns if ('site' in col.lower() or 'sex' in col.lower() or 'group' in col.lower() or main_df[col].dtype == 'object')]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"\nGrouping by field (@id/column): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field detected.")

    except Exception as e:
        print(f"EDA error: {e}")

## 5. Visualization

Let's visualize the distribution of the numeric variable and its relationship to a group field using Matplotlib and Seaborn, referencing all variables by their DataFrame column (which corresponds to `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_sets and possible_numeric_fields and possible_group_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Insufficient variable info for visualization.")

## 6. Conclusion

* Using the `mlcroissant` library and Croissant schema, we loaded and explored the FAIR^2 colorectal cancer survivors dataset entirely by referencing `@id` mappings for all record sets and fields.
* We performed data inspection, simple filtering, normalization, grouping, and visualized key distributions for further analysis.
* This approach supports reproducible and standards-based data science using the Croissant framework.